# Sesión 1 — Demo boto3
## Lo mismo que hicimos en la consola, ahora con código

> Corre esto en SageMaker Studio. Las credenciales del Learner Lab ya están
> configuradas en el ambiente — boto3 las detecta automáticamente, no
> necesitas escribir ninguna access key.

## 0. Setup

In [ ]:
import boto3
import pandas as pd

s3 = boto3.client('s3')

print('Conectado a S3 correctamente.')

---
## 1. Listar los buckets que ya existen en la cuenta

> Aquí debería aparecer el bucket que acabamos de crear en la consola.

In [ ]:
response = s3.list_buckets()

print('Buckets en esta cuenta:')
for bucket in response['Buckets']:
    print(' -', bucket['Name'])

---
## 2. Definir el bucket con el que vamos a trabajar

> ⚠️ Cambia el nombre por el bucket que creaste tú en la consola —
> recuerda que el nombre es único globalmente, así que el tuyo
> seguramente es distinto al de tus compañeros.

In [ ]:
BUCKET = 'reviews-demo-XXXX'  # <- reemplaza con tu bucket real

---
## 3. Ver lo que ya subimos por consola

> `reviews.csv` ya debería estar ahí — lo subimos con drag & drop
> hace un momento. Esto confirma que consola y código apuntan
> exactamente al mismo lugar.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET)

print(f'Contenido de {BUCKET}:')
for obj in response.get('Contents', []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

---
## 4. Subir un archivo nuevo — ahora con código, no con clics

> Vamos a crear la carpeta `processed/` sin haber tocado la consola.
> En S3 las "carpetas" no existen realmente — son solo el prefijo
> del nombre del archivo (el `key`).

In [ ]:
# Creamos un archivo de ejemplo localmente para subirlo
with open('nota.txt', 'w') as f:
    f.write('Este archivo se subió con boto3, no con clics.')

s3.upload_file(
    Filename='nota.txt',
    Bucket=BUCKET,
    Key='processed/nota.txt'
)

print('Archivo subido. Ve a refrescar la consola y búscalo en processed/.')

> 👉 **Pausa aquí y regresa a la pestaña de la consola** — dale refresh
> al bucket y muestra que `processed/nota.txt` ya está ahí, sin que
> nadie le diera clic a "Create folder".

---
## 5. Leer el CSV directamente desde S3 con pandas

> No hace falta descargar el archivo a disco local primero.

In [ ]:
ruta_s3 = f's3://{BUCKET}/raw/reviews.csv'

df = pd.read_csv(ruta_s3)

print(f'Shape: {df.shape}')
df.head()

---
## 6. EDA básico — primer vistazo al dataset del caso de uso

> Este es el dataset de reseñas que vamos a ir enriqueciendo a lo
> largo de las demos del curso (NLP en la semana 6, GenAI en la 8).

In [ ]:
print('Distribución de ratings:')
print(df['rating'].value_counts().sort_index())
print()
print('Productos únicos:', df['product'].nunique())
print('% de compras verificadas:', round(df['verified_purchase'].mean() * 100, 1), '%')

In [ ]:
import matplotlib.pyplot as plt

VIOLET = '#6C4FF6'

fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
df['rating'].value_counts().sort_index().plot(
    kind='bar', color=VIOLET, edgecolor='white', ax=ax
)
ax.set_xlabel('Rating')
ax.set_ylabel('Número de reseñas')
ax.set_title('Distribución de ratings — dataset de reseñas')
plt.tight_layout()
plt.show()

---
## 7. Guardar el resultado del EDA de vuelta en S3

> Cerramos el círculo: leímos de `raw/`, procesamos algo simple,
> y lo guardamos en `processed/` — la estructura que vimos en el apunte.

In [ ]:
resumen = df.groupby('product')['rating'].mean().round(2).reset_index()
resumen.columns = ['product', 'avg_rating']
resumen.to_csv('resumen_por_producto.csv', index=False)

s3.upload_file(
    Filename='resumen_por_producto.csv',
    Bucket=BUCKET,
    Key='processed/resumen_por_producto.csv'
)

print('Resumen guardado en processed/resumen_por_producto.csv')
resumen.sort_values('avg_rating', ascending=False)

---
## Cierre

Hoy hicimos, con 4 líneas de código cada vez, exactamente lo mismo que
hicimos con clics en la consola:

| Acción | Consola | boto3 |
|---|---|---|
| Crear bucket | Create bucket | `s3.create_bucket(...)` |
| Subir archivo | Drag & drop | `s3.upload_file(...)` |
| Ver contenido | Navegar visualmente | `s3.list_objects_v2(...)` |

**La próxima sesión:** SageMaker Studio a fondo — vamos a preprocesar
este mismo dataset y dejarlo listo para entrenar un modelo.